In [1]:
from dotenv import load_dotenv; load_dotenv()

True

In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3'

In [3]:
import torch
torch.cuda.empty_cache()

In [4]:
!nvidia-smi

Mon Mar 16 23:09:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:A4:00.0 Off |                    0 |
| N/A   28C    P0             69W /  700W |       4MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

### Dataset Generation

In [6]:
from datasets import load_dataset

TRAIN_SIZE = 2000
dataset = load_dataset("allenai/tulu-3-sft-mixture", split="train", streaming=True)
dataset.shuffle(seed=42)
train_dataset = []
for i, x in enumerate(dataset):
    if i >= TRAIN_SIZE:
        break
    train_dataset.append(dict(messages=x['messages']))

train_dataset[0]

{'messages': [{'content': 'Create a snippet of Terraform HCL code that create an AWS autoscaling group, and an ALB in front to expose an application to internet.',
   'role': 'user'},
  {'content': 'Sure, here\'s an example Terraform HCL code that creates an AWS Autoscaling Group and an Application Load Balancer to expose an application to the internet:\n``` \n# Configure the AWS provider\nprovider "aws" {\n  region = "us-east-1"\n}\n\n# Create a security group to allow traffic to the ALB\nresource "aws_security_group" "alb_sg" {\n  name_prefix = "alb_sg"\n  ingress {\n    from_port = 80\n    to_port = 80\n    protocol = "tcp"\n    cidr_blocks = ["0.0.0.0/0"]\n  }\n}\n\n# Create an ALB and target group\nresource "aws_lb" "alb" {\n  name               = "example-alb"\n  internal           = false\n  load_balancer_type = "application"\n\n  subnets = ["subnet-12345678", "subnet-87654321"]\n\n  security_groups = [aws_security_group.alb_sg.id]\n\n  tags = {\n    Environment = "production"\n

In [7]:
import os

max_seq_length = 1024
STUDENT_MODEL_NAME = "./knowledge-ingestion-test/model/checkpoint-178"
os.path.exists(STUDENT_MODEL_NAME)


True

In [8]:
from peft import PeftConfig, PeftModel

config = PeftConfig.from_pretrained(STUDENT_MODEL_NAME)
config.base_model_name_or_path

'unsloth/Qwen3-4B-Instruct-2507'

In [9]:
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
base_model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (

In [10]:
# student_model = base_model
student_model = PeftModel.from_pretrained(
    base_model,
    STUDENT_MODEL_NAME,
    is_trainable=True,   # True if you want to keep training
)
student_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_p

In [11]:
student_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2560, padding_idx=151654)
        (layers): ModuleList(
          (0-35): 36 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_p

In [12]:
for name, param in student_model.named_parameters():
    if "lora" in name:
        assert param.requires_grad == True
    else:
        assert param.requires_grad == False



In [13]:
device = next(student_model.parameters()).device
device

device(type='cuda', index=0)

In [14]:
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

In [15]:
optimizer = torch.optim.AdamW(student_model.parameters(), lr=1e-5)

In [16]:
@torch.no_grad()
def sample_from_student(messages, max_new_tokens=256, temperature=1.0, top_p=1.0):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, padding_side="left").to(student_model.device)
    prompt_len = inputs["input_ids"].shape[1]

    sampled = student_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return sampled, prompt_len

messages = [
    [{'role': 'user', 'content': 'Hello, how are you?'}],
    [{'role': 'user', 'content': 'yo whats the weather in tokyo?'}],
]

sampled, prompt_len = sample_from_student(messages)
print(tokenizer.decode(sampled))

['<|vision_pad|><|vision_pad|><|vision_pad|><|im_start|>user\nHello, how are you?<|im_end|>\n<|im_start|>assistant\n-surprise- How’s it going, buddy?<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>', '<|im_start|>user\nyo whats the weather in tokyo?<|im_end|>\n<|im_start|>assistant\n<tool_call>\n\n</think>\n\nThe weather in Tokyo is currently raining on the streets, which is not an option for the AI assistant to describe since it cannot function in reality.<|im_end|>']


In [17]:
import requests
def generate_teacher_logprobs(input_ids, attention_mask):
    input_ids = input_ids.tolist(); attention_mask = attention_mask.tolist()
    response = requests.post("http://localhost:8000/logprobs", json={"input_ids": input_ids, "attention_mask": attention_mask})
    return torch.tensor(response.json()["logprobs"])


In [18]:
from tqdm import tqdm

# One Step
def opd_step(messages, micro_batch_size=1):
    optimizer.zero_grad()
    student_model.zero_grad()

    final_loss = 0
    final_mean_advantage = 0
    final_mean_reverse_kl = 0

    for i in tqdm(range(0, len(messages), micro_batch_size), desc="OPD Step", leave=False):
        # generate rollout
        sampled, prompt_len = sample_from_student(messages[i:i+micro_batch_size])

        full_ids = sampled
        full_ids = full_ids.to(device)
        target_ids = full_ids[:, 1:]
        attn = (full_ids != tokenizer.pad_token_id)

        # get student logprobs
        out = student_model(input_ids=full_ids, attention_mask=attn)
        student_logprobs = F.log_softmax(out.logits[:, :-1, :], dim=-1)
        student_lp = student_logprobs.gather(dim=-1, index=target_ids.unsqueeze(-1)).squeeze(-1)

        # get teacher logprobs
        teacher_lp = generate_teacher_logprobs(full_ids, attn)

        # get advantage
        with torch.no_grad():
            advantage = (teacher_lp.to(student_lp.device) - student_lp.detach()).clamp(-5, 5)

        # mask
        mask = torch.ones_like(target_ids) & (target_ids != tokenizer.eos_token_id) & (target_ids != tokenizer.pad_token_id)
        mask[:, :prompt_len-1] = 0
        mask = mask.to(student_lp.device)

        # get loss
        loss = -(mask * student_lp * advantage).sum() / mask.sum().clamp(min=1)
        loss.backward()

        # stats
        final_loss += loss.item()
        final_mean_advantage += ((mask*advantage).sum() / mask.sum().clamp(min=1)).item()
        final_mean_reverse_kl += -((mask*advantage).sum() / mask.sum().clamp(min=1)).item()

    # backprop
    total_norm = torch.nn.utils.clip_grad_norm_(student_model.parameters(), 1.0)
    optimizer.step()

    # stats
    final_loss = final_loss / (len(messages) / micro_batch_size)
    final_mean_advantage = final_mean_advantage / (len(messages) / micro_batch_size)
    final_mean_reverse_kl = final_mean_reverse_kl / (len(messages) / micro_batch_size)

    return {
        'loss': final_loss,
        'total_norm': total_norm.item(),
        'mean_advantage': final_mean_advantage,
        'mean_reverse_kl': final_mean_reverse_kl,
    }

In [ ]:
BATCH_SIZE = 64
MICRO_BATCH_SIZE = 8

import wandb
# Start a run
wandb.init(
    project="sunday-auto-customize",
    name="demo-run-opd",
)

END = len(train_dataset)
# END = 128
for epoch in range(5):
    pbar = tqdm(range(0, END, BATCH_SIZE))
    for idx in pbar:
        batch = [x['messages'][:-1] for x in train_dataset[idx:idx+BATCH_SIZE]]
        stats = opd_step(batch, MICRO_BATCH_SIZE)
        wandb.log(stats)
        pbar.set_postfix(stats)
    pbar.close()

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ronny21 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


 78%|█████████████████████████████▋        | 25/32 [53:22<14:15, 122.23s/it, loss=-1.63, total_norm=23.2, mean_advantage=-0.437, mean_reverse_kl=0.437]

In [ ]:
OUTPUT_DIR = './knowledge-ingestion-test/post_trained_model/'

# save model
student_model.save_pretrained(OUTPUT_DIR)
